In [2]:
import asyncio
import random
import logging
from typing import Dict, Any

# -----------------------------
# Logging setup
# -----------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

# -----------------------------
# Base Agent
# -----------------------------
class BaseAgent:
    def __init__(self, name: str):
        self.name = name

    async def handle(self, message: Dict[str, Any]) -> Dict[str, Any]:
        raise NotImplementedError


# -----------------------------
# Agents
# -----------------------------
class PortfolioAgent(BaseAgent):
    async def handle(self, message):
        await asyncio.sleep(1)

        # simulate failure
        if random.random() < 0.2:
            raise Exception("Portfolio analysis failed")

        return {
            "agent": self.name,
            "output": f"Portfolio analyzed for {message['input']['user']}",
            "data": {"value": random.randint(10000, 50000)}
        }


class RiskAgent(BaseAgent):
    async def handle(self, message):
        await asyncio.sleep(1)

        if random.random() < 0.2:
            raise Exception("Risk evaluation failed")

        return {
            "agent": self.name,
            "output": "Risk evaluated",
            "data": {"risk_score": random.uniform(0, 1)}
        }


class ExternalAgent(BaseAgent):
    async def handle(self, message):
        await asyncio.sleep(1.5)

        if random.random() < 0.3:
            raise Exception("External agent timeout")

        return {
            "agent": self.name,
            "output": "External insights generated",
            "data": {"insight": "Diversify into bonds"}
        }


# -----------------------------
# Orchestrator
# -----------------------------
class Orchestrator:
    def __init__(self, agents: Dict[str, BaseAgent]):
        self.agents = agents

    async def safe_call(self, agent: BaseAgent, message: Dict[str, Any], retries=2):
        for attempt in range(retries + 1):
            try:
                logging.info(f"Calling {agent.name} (attempt {attempt+1})")
                result = await agent.handle(message)
                return result
            except Exception as e:
                logging.warning(f"{agent.name} failed: {e}")
        return {"agent": agent.name, "error": "failed after retries"}

    async def route(self, task: str, payload: Dict[str, Any]):
        message = {
            "task": task,
            "input": payload,
            "metadata": {"source": "orchestrator"}
        }

        logging.info(f"Starting task: {task}")

        # Step 1
        portfolio = await self.safe_call(self.agents["portfolio"], message)

        # Step 2
        risk = await self.safe_call(
            self.agents["risk"],
            {**message, "context": portfolio}
        )

        # Step 3
        external = await self.safe_call(
            self.agents["external"],
            {
                **message,
                "context": {
                    "portfolio": portfolio,
                    "risk": risk
                }
            }
        )

        final = {
            "portfolio": portfolio,
            "risk": risk,
            "external": external
        }

        logging.info("Task completed")
        return final


# -----------------------------
# Run
# -----------------------------
async def main():
    agents = {
        "portfolio": PortfolioAgent("PortfolioAgent"),
        "risk": RiskAgent("RiskAgent"),
        "external": ExternalAgent("ExternalAgent")
    }

    orchestrator = Orchestrator(agents)

    result = await orchestrator.route(
        task="analyze_user_portfolio",
        payload={"user": "Alice"}
    )

    print("\nFinal Output:\n", result)


await main()

2026-05-01 05:49:50,225 [INFO] Starting task: analyze_user_portfolio
2026-05-01 05:49:50,226 [INFO] Calling PortfolioAgent (attempt 1)
2026-05-01 05:49:51,226 [WARNING] PortfolioAgent failed: Portfolio analysis failed
2026-05-01 05:49:51,227 [INFO] Calling PortfolioAgent (attempt 2)
2026-05-01 05:49:52,228 [WARNING] PortfolioAgent failed: Portfolio analysis failed
2026-05-01 05:49:52,229 [INFO] Calling PortfolioAgent (attempt 3)
2026-05-01 05:49:53,230 [INFO] Calling RiskAgent (attempt 1)
2026-05-01 05:49:54,234 [WARNING] RiskAgent failed: Risk evaluation failed
2026-05-01 05:49:54,234 [INFO] Calling RiskAgent (attempt 2)
2026-05-01 05:49:55,235 [WARNING] RiskAgent failed: Risk evaluation failed
2026-05-01 05:49:55,236 [INFO] Calling RiskAgent (attempt 3)
2026-05-01 05:49:56,238 [INFO] Calling ExternalAgent (attempt 1)
2026-05-01 05:49:57,740 [INFO] Task completed



Final Output:
 {'portfolio': {'agent': 'PortfolioAgent', 'output': 'Portfolio analyzed for Alice', 'data': {'value': 20318}}, 'risk': {'agent': 'RiskAgent', 'output': 'Risk evaluated', 'data': {'risk_score': 0.9458656055479778}}, 'external': {'agent': 'ExternalAgent', 'output': 'External insights generated', 'data': {'insight': 'Diversify into bonds'}}}
